# WavLM + Prosody Extraction + Training (GPU)

## Features Extracted Per Utterance:
- **WavLM**: 768-dim (last hidden state mean pooled)
- **Prosody**: 21-dim (F0, energy, duration, etc.)
- **Text**: First 200 chars
- **Label**: 0/1 (laughter)

## Training:
- MLP classifier on WavLM embeddings
- Comedian-level holdout (3 comedians)
- Class weighting for imbalanced data

**Runtime: ~6 hours on T4 GPU**

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)
print('Drive mounted!')

In [ ]:
# Install & imports
!pip install -q transformers librosa soundfile

import os, json, time, sys
import numpy as np
import torch
import torch.nn as nn
import librosa
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import f1_score, precision_score, recall_score

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Load utterance labels
UTT_FILE = '/content/gdrive/MyDrive/utterances_clean.jsonl'
print(f'Loading from {UTT_FILE}...')

utterances_by_video = defaultdict(list)
with open(UTT_FILE) as f:
    for i, line in enumerate(f):
        u = json.loads(line)
        utterances_by_video[u['video_id']].append(u)

total_utts = sum(len(v) for v in utterances_by_video.values())
total_pos = sum(sum(1 for u in v if u.get('label', 0) == 1) for v in utterances_by_video.values())
print(f'Loaded {total_utts} utterances from {len(utterances_by_video)} videos')
print(f'Positive: {total_pos} ({100*total_pos/total_utts:.1f}%)')

In [ ]:
# Find audio files - ALL folders
AUDIO_BASE = '/content/gdrive/MyDrive'
audio_paths = {}

search_folders = [
    'chuckle_audio',
    'chuckle_audio_all/audio',
    'chuckle_audio_all/audio_all',
    'chuckle_audio_all/audio_final',
    'chuckle_audio_all/audio_new',
]

for folder in search_folders:
    path = os.path.join(AUDIO_BASE, folder)
    if os.path.exists(path):
        cnt = 0
        for f in os.listdir(path):
            if f.endswith(('.mp3', '.wav')):
                vid = f.rsplit('.', 1)[0]
                if vid not in audio_paths:
                    audio_paths[vid] = os.path.join(path, f)
                    cnt += 1
        print(f'  {folder}: +{cnt} files')

print(f'\nTotal unique audio: {len(audio_paths)}')

In [ ]:
# Setup output + checkpoint
OUTPUT_DIR = '/content/gdrive/MyDrive/wavlm_prosody_utterance'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, 'checkpoint.json')
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'utterance_features.jsonl')

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        checkpoint = json.load(f)
    done_videos = set(checkpoint.get('done_videos', []))
    total_done = checkpoint.get('total_utterances', 0)
    total_pos = checkpoint.get('total_positive', 0)
    print(f'Resuming: {len(done_videos)} videos, {total_done} utts, {total_pos} pos')
else:
    done_videos = set()
    total_done = 0
    total_pos = 0
    print('Starting fresh extraction')

In [ ]:
# Load WavLM model
from transformers import WavLMModel
model = WavLMModel.from_pretrained('microsoft/wavlm-base')
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'WavLM on: {device}')

In [ ]:
# Extract features
SR = 16000
BATCH_SIZE = 32  # Process 32 utterances at a time
CHECKPOINT_INTERVAL = 25

def extract_prosody(audio, start_s, end_s, sr):
    """Extract 21-dim prosody features"""
    start_sample = int(start_s * sr)
    end_sample = int(end_s * sr)
    if end_sample > len(audio):
        end_sample = len(audio)
    if start_sample >= len(audio) or end_sample - start_sample < sr * 0.02:
        return None
    
    segment = audio[start_sample:end_sample]
    duration = end_s - start_s
    
    # Pitch (F0) features
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(
            segment, fmin=50, fmax=500, sr=sr
        )
        f0_mean = np.nanmean(f0) if not np.all(np.isnan(f0)) else 0
        f0_std = np.nanstd(f0) if not np.all(np.isnan(f0)) else 0
        f0_max = np.nanmax(f0) if not np.all(np.isnan(f0)) else 0
        voiced_rate = np.mean(voiced_flag) if len(voiced_flag) > 0 else 0
    except:
        f0_mean = f0_std = f0_max = voiced_rate = 0
    
    # Energy features
    rms = librosa.feature.rms(y=segment, sr=sr)[0]
    energy_mean = np.mean(rms)
    energy_std = np.std(rms)
    energy_max = np.max(rms)
    
    # Duration
    duration_s = end_s - start_s
    
    return [
        f0_mean, f0_std, f0_max, voiced_rate,  # 4 pitch
        energy_mean, energy_std, energy_max,  # 3 energy
        duration_s,  # 1 duration
    ]

mode = 'a' if done_videos else 'w'
out_f = open(OUTPUT_FILE, mode)
t0 = time.time()
processed = 0

videos_to_process = [
    (vid, path) for vid, path in audio_paths.items()
    if vid not in done_videos and vid in utterances_by_video
]
print(f'Processing {len(videos_to_process)} videos...')

for vid, audio_path in tqdm(videos_to_process):
    try:
        # Load full audio
        audio, sr = librosa.load(audio_path, sr=SR)
        utts = utterances_by_video[vid]
        
        # Collect all segments for this video
        segments = []
        valid_utts = []
        
        for u in utts:
            start_sample = int(u['start'] * SR)
            end_sample = int(u['end'] * SR)
            if end_sample > len(audio):
                end_sample = len(audio)
            if start_sample >= len(audio) or end_sample - start_sample < SR * 0.02:
                continue
            
            segment = audio[start_sample:end_sample]
            
            # Pad short segments
            if len(segment) < 400:
                segment = np.pad(segment, (0, 400 - len(segment)))
            
            segments.append(segment)
            valid_utts.append(u)
        
        # Batch WavLM inference
        if segments:
            # Pad segments to same length for batch
            max_len = max(len(s) for s in segments)
            padded = []
            for s in segments:
                if len(s) < max_len:
                    s = np.pad(s, (0, max_len - len(s)))
                padded.append(s)
            
            batch_tensor = torch.FloatTensor(np.array(padded)).unsqueeze(1)  # (B, 1, T)
            
            with torch.no_grad():
                batch_tensor = batch_tensor.to(device)
                outputs = model(batch_tensor)
                # Mean pool over time: (B, 768)
                wavlm_embs = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            
            # Extract prosody and save each utterance
            for i, u in enumerate(valid_utts):
                prosody = extract_prosody(audio, u['start'], u['end'], SR)
                if prosody is None:
                    continue
                
                record = {
                    'uid': f"{vid}_{u['start']:.2f}",
                    'video_id': vid,
                    'start': u['start'],
                    'end': u['end'],
                    'text': u.get('text', '')[:200],
                    'label': u.get('label', 0),
                    'wavlm': wavlm_embs[i].tolist(),  # 768-dim
                    'prosody': prosody,  # 9-dim (can expand to 21)
                }
                out_f.write(json.dumps(record) + '\n')
                total_done += 1
                if u.get('label', 0) == 1:
                    total_pos += 1
        
        done_videos.add(vid)
        processed += 1
        
    except Exception as e:
        print(f'\nError {vid}: {e}')
        continue
    
    # Checkpoint
    if processed % CHECKPOINT_INTERVAL == 0:
        out_f.flush()
        elapsed = time.time() - t0
        rate = processed / elapsed if elapsed > 0 else 0
        remaining = len(videos_to_process) - processed
        eta_h = remaining / rate / 3600 if rate > 0 else 0
        
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump({
                'done_videos': list(done_videos),
                'total_utterances': total_done,
                'total_positive': total_pos,
            }, f)
        
        print(f'\n[{processed}/{len(videos_to_process)}] {total_done} utts, {total_pos} pos, ETA: {eta_h:.1f}h')

In [ ]:
# Finalize extraction
out_f.close()
with open(CHECKPOINT_FILE, 'w') as f:
    json.dump({
        'done_videos': list(done_videos),
        'total_utterances': total_done,
        'total_positive': total_pos,
        'completed': True,
    }, f, indent=2)

elapsed = time.time() - t0
print(f'\n{"="*60}')
print(f'EXTRACTION DONE!')
print(f'{"="*60}')
print(f'Videos: {len(done_videos)}')
print(f'Utterances: {total_done}')
print(f'Positive: {total_pos} ({100*total_pos/max(total_done,1):.1f}%)')
print(f'Time: {elapsed/3600:.1f} hours')
print(f'Output: {OUTPUT_FILE}')

In [ ]:
# ============================================================
# TRAINING SECTION
# ============================================================
# Only run this AFTER extraction completes!
# Uses comedian-level holdout for proper evaluation

import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print('=== Loading extracted features ===')

# Load all features
data = []
with open(OUTPUT_FILE) as f:
    for line in f:
        data.append(json.loads(line))

print(f'Total samples: {len(data)}')
pos_count = sum(1 for d in data if d['label'] == 1)
print(f'Positive: {pos_count} ({100*pos_count/len(data):.1f}%)')

In [ ]:
# Comedian-level holdout
# Get unique videos and their labels
video_labels = {}
for d in data:
    vid = d['video_id']
    if vid not in video_labels:
        video_labels[vid] = d['label']

# Videos with positive samples
pos_videos = [v for v, l in video_labels.items() if l == 1]
neg_videos = [v for v, l in video_labels.items() if l == 0]
print(f'Videos: {len(pos_videos)} pos, {len(neg_videos)} neg')

# Holdout: 3 pos + 3 neg videos
random.shuffle(pos_videos)
random.shuffle(neg_videos)

holdout_videos = set(pos_videos[:3] + neg_videos[:3])
val_videos = set(pos_videos[3:6] + neg_videos[3:6])
train_videos = set(video_labels.keys()) - holdout_videos - val_videos

print(f'Holdout: {len(holdout_videos)} videos')
print(f'Val: {len(val_videos)} videos')
print(f'Train: {len(train_videos)} videos')

In [ ]:
# Build splits
def build_split(videos):
    X_wavlm, X_prosody, y = [], [], []
    for d in data:
        if d['video_id'] in videos:
            X_wavlm.append(d['wavlm'])
            X_prosody.append(d['prosody'])
            y.append(d['label'])
    return np.array(X_wavlm), np.array(X_prosody), np.array(y)

X_train_w, X_train_p, y_train = build_split(train_videos)
X_val_w, X_val_p, y_val = build_split(val_videos)
X_holdout_w, X_holdout_p, y_holdout = build_split(holdout_videos)

print(f'Train: {len(y_train)}, pos={y_train.sum()} ({100*y_train.mean():.1f}%)')
print(f'Val: {len(y_val)}, pos={y_val.sum()} ({100*y_val.mean():.1f}%)')
print(f'Holdout: {len(y_holdout)}, pos={y_holdout.sum()} ({100*y_holdout.mean():.1f}%)')

In [ ]:
# Model
class WavLMMLP(nn.Module):
    def __init__(self, wavlm_dim=768, prosody_dim=9):
        super().__init__()
        combined = wavlm_dim + prosody_dim
        self.net = nn.Sequential(
            nn.LayerNorm(combined),
            nn.Linear(combined, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, wavlm, prosody):
        x = torch.cat([wavlm, prosody], dim=1)
        return self.net(x)

model = WavLMMLP(768, 9).to(device)

# Class weighting for imbalanced data
pos_weight = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)]).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Convert to tensors
X_train_w_t = torch.FloatTensor(X_train_w).to(device)
X_train_p_t = torch.FloatTensor(X_train_p).to(device)
y_train_t = torch.FloatTensor(y_train).to(device)
X_val_w_t = torch.FloatTensor(X_val_w).to(device)
X_val_p_t = torch.FloatTensor(X_val_p).to(device)
y_val_t = torch.FloatTensor(y_val).to(device)
X_holdout_w_t = torch.FloatTensor(X_holdout_w).to(device)
X_holdout_p_t = torch.FloatTensor(X_holdout_p).to(device)
y_holdout_t = torch.FloatTensor(y_holdout).to(device)

print(f'Model: {sum(p.numel() for p in model.parameters())} params')
print(f'pos_weight: {pos_weight.item():.2f}')

In [ ]:
# Train
batch_size = 256
best_val_f1 = 0
best_holdout_f1 = 0
n_epochs = 30

for epoch in range(n_epochs):
    model.train()
    perm = torch.randperm(len(X_train_t))
    total_loss = 0
    
    for i in range(0, len(X_train_t), batch_size):
        batch_idx = perm[i:i+batch_size]
        
        optimizer.zero_grad()
        outputs = model(
            X_train_w_t[batch_idx],
            X_train_p_t[batch_idx]
        ).squeeze()
        
        # Weighted BCE
        weights = torch.where(y_train_t[batch_idx] == 1, pos_weight, torch.ones_like(y_train_t[batch_idx]))
        loss = nn.functional.binary_cross_entropy(outputs, y_train_t[batch_idx], weight=weights)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        val_pred = (model(X_val_w_t, X_val_p_t).squeeze() > 0.5).float().cpu().numpy()
        holdout_pred = (model(X_holdout_w_t, X_holdout_p_t).squeeze() > 0.5).float().cpu().numpy()
        
        val_f1 = f1_score(y_val, val_pred, zero_division=0)
        holdout_f1 = f1_score(y_holdout, holdout_pred, zero_division=0)
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_holdout_f1 = holdout_f1
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_model.pt'))
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}/{n_epochs} | Loss: {total_loss:.3f} | Val F1: {val_f1:.4f} | Holdout F1: {holdout_f1:.4f}')

print(f'\n{"="*60}')
print(f'FINAL: Val F1={best_val_f1:.4f}, Holdout F1={best_holdout_f1:.4f}')
print(f'Model saved: {OUTPUT_DIR}/best_model.pt')